# TODO: Refactor this notebook

In [1]:
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pycocotools.mask as mask
from PIL import Image
import itertools
import os
import scipy.ndimage as ndimage


In [17]:
os.path.exists('../../Global_Outputs/wormswin_FineTune_filtered_artifacts_cnt_wormswin_csb-1/wormswin_FineTune_filtered_artifacts_cnt_wormswin_csb-1_predicted_annotations.pkl')

True

In [ ]:
import pickle
pickle_file = '../../Global_Outputs/wormswin_FineTune_filtered_artifacts_cnt_wormswin_csb-1/wormswin_FineTune_filtered_artifacts_cnt_wormswin_csb-1_predicted_annotations.pkl'
with open(pickle_file, 'rb') as f:
    data = pickle.load(f)

In [19]:
def convert_wormswin_to_coco(result_pkl, output_json="wormswin_coco.json", score_threshold=0.5):
    with open(result_pkl, "rb") as f:
        data = pickle.load(f)

    coco_data = {"images": [], "annotations": [], "categories": [{"id": 1, "name": "CNT"}]}
    annotation_id = 1

    for image_id, img_data in enumerate(data):
        detection_scores = img_data[0][0]
        detection_masks = img_data[1][0]

        filename = f"image_{image_id}.jpg"
        height, width = detection_masks[0]["size"]

        coco_data["images"].append({"id": image_id + 1, "file_name": filename, "width": width, "height": height})

        for det_idx, det_score in enumerate(detection_scores):
            score = det_score[4]
            if score < score_threshold:
                continue

            rle_mask = detection_masks[det_idx]
            decoded_mask = mask.decode(rle_mask).astype(np.uint8)

            contours, _ = cv2.findContours(decoded_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            segmentation = []
            bbox_coords = []

            for contour in contours:
                if len(contour) < 3:
                    continue
                segmentation.append(contour.flatten().tolist())
                x, y, w, h = cv2.boundingRect(contour)
                bbox_coords.append((x, y, x + w, y + h))

            if bbox_coords:
                x_min = min([x for (x, _, _, _) in bbox_coords])
                y_min = min([y for (_, y, _, _) in bbox_coords])
                x_max = max([x_w for (_, _, x_w, _) in bbox_coords])
                y_max = max([y_h for (_, _, _, y_h) in bbox_coords])
                bbox = [x_min, y_min, x_max - x_min, y_max - y_min]
            else:
                continue

            area = cv2.contourArea(np.vstack(contours))

            annotation = {"id": annotation_id, "image_id": image_id + 1, "category_id": 1, "segmentation": segmentation,
                          "bbox": bbox, "iscrowd": 0, "area": area, "score": float(score)}
            coco_data["annotations"].append(annotation)
            annotation_id += 1

    with open(output_json, "w") as json_file:
        json.dump(coco_data, json_file, indent=4)

    print(f"COCO JSON saved to {output_json}")



In [20]:
def visualize_wormswin(image_index, result_pkl, img_path, gt_json):
    with open(result_pkl, "rb") as f:
        data = pickle.load(f)
    
    with open(gt_json, "r") as f:
        gt_data = json.load(f)
    
    image_file = next(img["file_name"] for img in gt_data["images"] if img["id"] == image_index+1)
    img = Image.open(os.path.join(img_path, image_file)).convert("RGBA")
    
    img_scores = data[image_index][0][0]
    img_masks = [mask.decode(m["segmentation"]) for m in data[image_index][1][0]]
    
    gt_masks = [mask.decode(anno["segmentation"]) for anno in gt_data["annotations"] if anno["image_id"] == image_index+1]
    
    fig, ax = plt.subplots(1, 2, figsize=(15, 8))
    
    # Plot GT
    ax[0].imshow(img)
    ax[0].set_title("Ground Truth")
    
    for gt_mask in gt_masks:
        gt_contours, _ = cv2.findContours(gt_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in gt_contours:
            ax[0].plot(contour[:, 0, 0], contour[:, 0, 1], color='r', linewidth=2)
    
    # Plot Predictions
    ax[1].imshow(img)
    ax[1].set_title("Predictions")
    
    for idx, pred_mask in enumerate(img_masks):
        pred_contours, _ = cv2.findContours(pred_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in pred_contours:
            ax[1].plot(contour[:, 0, 0], contour[:, 0, 1], color='b', linewidth=2)
            ax[1].text(contour[0, 0, 0], contour[0, 0, 1], f"{img_scores[idx][4]:.2f}", color='yellow', fontsize=8)
    
    plt.show()
    print(f"Displayed image {image_file} with predictions and GT.")

In [23]:
img_path = r"..\..\..\..\Data\annotations_filtered_artifacts\test\images"
output_gt_json_path = r"..\..\..\..\Data\annotations_filtered_artifacts\test\COCO_mask\annotations.json"
image_index = 0
visualize_wormswin(image_index, pickle_file, img_path, output_gt_json_path)

KeyError: 'segmentation'